In [21]:
import pandas as pd
from openai import OpenAI
from sklearn.metrics import accuracy_score

# 1. Veri Hazırlığı
df = pd.read_csv("final_train_data.csv").sample(50, random_state=42)

def create_ultimate_prompt(row):
    # Few-shot örnekleri: Modele rehberlik etmesi için 2 gerçek senaryoyu kurallarla veriyoruz
    return f"""
    Sen bir kaza analiz uzmanısın. Yolcuların verilerini inceleyerek transfer olasılığını hesapla.

    BİLGİ REHBERİ:
    - ÖRNEK 1: Kriyojenik uykuda olan (True) ve harcaması 0 olan yolcuların transfer olma olasılığı çok yüksektir. (SONUÇ: TRUE)
    - ÖRNEK 2: Uyanık olan (False), Spa ve VRDeck'te binlerce birim harcayan yolcuların transfer olma olasılığı düşüktür. (SONUÇ: FALSE)

    YENİ ANALİZ EDİLECEK YOLCU:
    - Kriyojenik Uyku: {'Evet' if row.get('CryoSleep') else 'Hayır'}
    - Memleket: {row.get('HomePlanet')}
    - Yaş: {row.get('Age')}
    - Toplam Harcama (Spa, VR, Oda Servisi): {row.get('RoomService', 0) + row.get('Spa', 0) + row.get('VRDeck', 0)}

    Analiz: Bu yolcunun durumunu yukarıdaki uzman kuralları ve örneklerle kıyaslayarak bir karara var.
    Lütfen SADECE en sonda 'SONUÇ: TRUE' veya 'SONUÇ: FALSE' yaz.
    """

df['Ultimate_Prompt'] = df.apply(create_ultimate_prompt, axis=1)
# client = OpenAI(api_key="api-key")

# Sunumun için karşılaştıracağımız iki ana model
modeller = ["gpt-3.5-turbo", "gpt-4"]

for model_adi in modeller:
    tahminler = []
    print(f"{model_adi} (Uzman + Few-Shot) test ediliyor...")

    for prompt in df['Ultimate_Prompt']:
        try:
            response = client.chat.completions.create(
                model=model_adi,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            cevap = response.choices[0].message.content.strip().upper()

            if "TRUE" in cevap: tahminler.append(True)
            elif "FALSE" in cevap: tahminler.append(False)
            else: tahminler.append(None)
        except:
            tahminler.append(None)

    col_name = f'{model_adi}_Expert_Prediction'
    df[col_name] = tahminler
    temiz = df.dropna(subset=[col_name, 'Transported'])
    acc = accuracy_score(temiz['Transported'].astype(bool), temiz[col_name].astype(bool))

    # Sunumda gpt-3.5-turbo için doğrudan GPT-3.5 ismini kullanabilirsin
    gosterim_adi = "GPT-3.5" if "3.5" in model_adi else "GPT-4"
    print(f"--- {gosterim_adi} Yeni Uzman Skor: %{acc * 100:.2f}")

print("\nKarşılaştırmalı test bitti! Sonuçları makale değerleriyle eşleştirebiliriz.")

gpt-3.5-turbo (Uzman + Few-Shot) test ediliyor...
--- GPT-3.5 Yeni Uzman Skor: %54.00
gpt-4 (Uzman + Few-Shot) test ediliyor...
--- GPT-4 Yeni Uzman Skor: %68.00

Karşılaştırmalı test bitti! Sonuçları makale değerleriyle eşleştirebiliriz.


In [24]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from scipy.optimize import differential_evolution

# 1. ÖRNEK VERİ HAZIRLIĞI (Bunu kendi gerçek tahmin veri çerçevenle değiştireceksin)
df = pd.read_csv("submission_kesin_cozum.csv")
# Tüm tabloyu matematiksel formata (1 ve 0) çevirme garantisi
df = df.astype(int)
y_true = df['Transported']

# Senaryo Grupları
trad_models = ['XGBoost', 'LightGBM', 'RandomForest']
all_models = ['XGBoost', 'LightGBM', 'RandomForest', 'GPT4', 'GPT35']

# --- METOD 1: KLASİK OYLAMA (MAJORITY VOTING) ---
def majority_voting(predictions_df):
    # Satır bazında toplamı alıp, model sayısının yarısından büyükse 1, değilse 0 yap
    return (predictions_df.sum(axis=1) > (predictions_df.shape[1] / 2)).astype(int)

# --- METOD 2: EVRİMSEL HESAPLAMA İLE OPTİMİZASYON (EVOLUTIONARY WEIGHTING) ---
def optimize_weights(predictions_df, y_true):
    preds_matrix = predictions_df.values
    n_models = preds_matrix.shape[1]

    # Optimizasyon için Kayıp (Loss) Fonksiyonu: 1 - Accuracy
    def loss_function(weights):
        # Ağırlıklı oylama
        weighted_sum = np.dot(preds_matrix, weights)
        threshold = np.sum(weights) / 2
        ensemble_pred = (weighted_sum > threshold).astype(int)
        return 1.0 - accuracy_score(y_true, ensemble_pred)

    # Evrimsel algoritma ayarları (Her modelin ağırlığı 0 ile 1 arasında olabilir)
    bounds = [(0.0, 1.0) for _ in range(n_models)]

    # Differential Evolution (Meta-Sezgisel) algoritmasını koştur
    result = differential_evolution(loss_function, bounds, strategy='best1bin', popsize=15, mutation=(0.5, 1), recombination=0.7, seed=42)

    best_weights = result.x
    best_accuracy = 1.0 - result.fun
    return best_weights, best_accuracy

# ================= SONUÇLARI HESAPLAMA =================

print("=== SENARYO A: SADECE GELENEKSEL MODELLER ===")
trad_voting_pred = majority_voting(df[trad_models])
print(f"1. Klasik Oylama (Majority Voting) Doğruluğu: %{accuracy_score(y_true, trad_voting_pred)*100:.2f}")

trad_weights, trad_evo_acc = optimize_weights(df[trad_models], y_true)
print(f"2. Evrimsel Hesaplama (Meta-Sezgisel) Doğruluğu: %{trad_evo_acc*100:.2f}")
print(f"   Bulunan Optimum Ağırlıklar: XGBoost({trad_weights[0]:.2f}), LightGBM({trad_weights[1]:.2f}), RF({trad_weights[2]:.2f})\n")

print("=== SENARYO B: GELENEKSEL MODELLER + BÜYÜK DİL MODELLERİ (LLM) ===")
all_voting_pred = majority_voting(df[all_models])
print(f"1. Klasik Oylama (Majority Voting) Doğruluğu: %{accuracy_score(y_true, all_voting_pred)*100:.2f}")

all_weights, all_evo_acc = optimize_weights(df[all_models], y_true)
print(f"2. Evrimsel Hesaplama (Meta-Sezgisel) Doğruluğu: %{all_evo_acc*100:.2f}")
print(f"   Bulunan Optimum Ağırlıklar: XGB({all_weights[0]:.2f}), LGBM({all_weights[1]:.2f}), RF({all_weights[2]:.2f}), GPT4({all_weights[3]:.2f}), GPT35({all_weights[4]:.2f})")

=== SENARYO A: SADECE GELENEKSEL MODELLER ===


KeyError: "None of [Index(['XGBoost', 'LightGBM', 'RandomForest'], dtype='str')] are in the [columns]"